# VISUALIZE AUGMENTATION OUTPUT

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import io
from PIL import Image
from typing import Any

from emma import AugmentedDataset, EmmaDataset
from emma.augment import HDF5Source
import hdf5plugin

## > DATASET

In [ ]:
DATASET_ROOT = Path("PATH/TO/EMMa")
AUGMENTATION_PATH = Path("PATH/TO/GLP-H5-FILE")
SOURCE_NAME = "glp_depth"

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Please set DATASET_ROOT to a valid EMMa dataset directory. Got: {DATASET_ROOT}"
    )

if not AUGMENTATION_PATH.is_file():
    raise FileNotFoundError(
        f"Please set AUGMENTATION_PATH to a valid augmentation file. Got: {AUGMENTATION_PATH}"
    )

In [ ]:
ds = EmmaDataset(DATASET_ROOT)

In [ ]:
depth_maps = HDF5Source(SOURCE_NAME, AUGMENTATION_PATH, mode="r")

In [ ]:
ads = AugmentedDataset(ds).register(depth_maps)

## > VISUALIZE

In [ ]:
num_samples = 10
random_seed = 67
random_sample = True

In [ ]:
augmented_asins = list(ads.source(SOURCE_NAME).keys())

if random_sample:
    rng = np.random.default_rng(random_seed)
    selected_asins = rng.choice(augmented_asins, size=min(num_samples, len(augmented_asins)), replace=False).tolist()
else:
    selected_asins = augmented_asins[:num_samples]

In [ ]:
def first_image_rgb(ds: EmmaDataset, rec: dict[str, Any]) -> Image.Image:
    image_dict = ds.get_images(rec)
    _, first_img_bytes = next(iter(image_dict.items()))
    return Image.open(io.BytesIO(first_img_bytes)).convert("RGB")


def unpack_depth(payload: Any) -> tuple[np.ndarray, dict[str, Any]]:
    depth = np.asarray(payload["pred"], dtype=np.float32)
    if "scale" in payload.dtype.names:
        depth /= float(payload["scale"])

    return np.squeeze(depth)


def depth_to_pil(depth: np.ndarray) -> Image.Image:
    depth_vis = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    depth_vis = (depth_vis * 255).astype(np.uint8)
    return Image.fromarray(depth_vis)

In [ ]:
for idx, asin in enumerate(selected_asins):
    rec = ads.get_record(asin)
    image = first_image_rgb(ds, rec)

    payload = ads.get_augmentation(SOURCE_NAME, asin)
    depth = unpack_depth(payload)
    depth_image = depth_to_pil(depth)

    image.show()
    depth_image.show()

    print("DEPTH MIN: ", depth.min())
    print("DEPTH MAX: ", depth.max())